# Simple Baseline

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time

from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module

# ============================================================
# PATHS
# ============================================================
LOAD_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv"
WEATHER_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\meteo_imputed.csv"
DATES_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json"

OUTPUT_DIR = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\moirai2_5min_SB"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
TARGET_COL = "avg_power"

WEATHER_COLS = [
    "Temp__C",
    "Hum__",
    "AvgWindSpeed_M_s",
    "Rain_Mm",
    "SolarRad_W_m_2",
]

PRED_LEN = 288
CONTEXT_LEN = 288 * 30

FREQ = "5min"
BATCH_SIZE = 1


MODEL_NAME = "Salesforce/moirai-2.0-R-small"

# ============================================================
# LOAD DATA
# ============================================================
load_df = pd.read_csv(LOAD_PATH)
weather_df = pd.read_csv(WEATHER_PATH)

load_df.columns = load_df.columns.str.strip()
weather_df.columns = weather_df.columns.str.strip()

weather_df = weather_df.rename(columns={"Date_Time": "timestamp"})

load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])

load_df = load_df.set_index("timestamp")
weather_df = weather_df.set_index("timestamp")

load_5min = load_df.resample(FREQ).mean()
weather_5min = weather_df.resample(FREQ).mean()

df = load_5min.join(weather_5min, how="inner").reset_index()
df = df.sort_values("timestamp").reset_index(drop=True)
df = df.dropna(subset=[TARGET_COL] + WEATHER_COLS).reset_index(drop=True)

print("Data range:", df["timestamp"].min(), "to", df["timestamp"].max())
print(df[[TARGET_COL] + WEATHER_COLS].head())

# ============================================================
# FORECAST DATES
# ============================================================
with open(DATES_PATH, "r") as f:
    forecast_dates = json.load(f)

# ============================================================
# LOAD MOIRAI2
# ============================================================
model = Moirai2Forecast(
    module=Moirai2Module.from_pretrained(MODEL_NAME),
    prediction_length=PRED_LEN,
    context_length=CONTEXT_LEN,
    target_dim=1,
    feat_dynamic_real_dim=0,                       # no known future covariates
    past_feat_dynamic_real_dim=len(WEATHER_COLS),  # past weather only
)

predictor = model.create_predictor(batch_size=BATCH_SIZE)

# ============================================================
# HELPERS
# ============================================================
def extract_forecast_values(forecast):
    """
    Robustly extract point forecast from Moirai/GluonTS forecast object.
    Prefer median/0.5 quantile when available.
    """

    if hasattr(forecast, "quantile"):
        try:
            return np.asarray(forecast.quantile(0.5))[:PRED_LEN]
        except Exception:
            pass

        try:
            return np.asarray(forecast.quantile("0.5"))[:PRED_LEN]
        except Exception:
            pass

    if hasattr(forecast, "median"):
        return np.asarray(forecast.median)[:PRED_LEN]

    if hasattr(forecast, "mean"):
        return np.asarray(forecast.mean)[:PRED_LEN]

    if hasattr(forecast, "samples"):
        return np.median(np.asarray(forecast.samples), axis=0)[:PRED_LEN]

    raise ValueError(f"Cannot extract forecast values from object: {type(forecast)}")


def make_moirai_dataset(history_df):
    target = history_df[TARGET_COL].astype(np.float32).values

    past_weather = history_df[WEATHER_COLS].astype(np.float32).values.T
    # shape: [num_past_covariates, time]

    entry = {
        "start": history_df["timestamp"].iloc[0],
        "target": target,
        "past_feat_dynamic_real": past_weather,
    }

    return ListDataset([entry], freq=FREQ)

# ============================================================
# FORECAST FUNCTION
# ============================================================
def run_one_forecast(season, day_name, test_start):

    test_start = pd.Timestamp(test_start)
    test_end = test_start + pd.Timedelta(days=1)

    history_df = df[df["timestamp"] < test_start].tail(CONTEXT_LEN).copy()

    future_df = df[
        (df["timestamp"] >= test_start)
        & (df["timestamp"] < test_end)
    ].copy()

    if len(history_df) == 0:
        print(f"Skipping {season}-{day_name}: no history before {test_start}")
        return None

    if len(history_df) < CONTEXT_LEN:
        print(
            f"Warning {season}-{day_name}: expected context {CONTEXT_LEN}, "
            f"got {len(history_df)}"
        )

    if len(future_df) != PRED_LEN:
        print(f"Skipping {season}-{day_name}: expected {PRED_LEN}, got {len(future_df)}")
        return None

    dataset = make_moirai_dataset(history_df)

    forecasts = list(predictor.predict(dataset))

    if len(forecasts) != 1:
        raise RuntimeError(f"Expected 1 forecast, got {len(forecasts)}")

    pred_values = extract_forecast_values(forecasts[0])

    out = future_df[["timestamp", TARGET_COL]].copy()
    out = out.rename(columns={TARGET_COL: "actual"})

    out["prediction"] = pred_values
    out["season"] = season
    out["day"] = day_name

    return out

# ============================================================
# RUN ALL DAYS
# ============================================================
all_results = []
start_time = time.time()

for season, days in forecast_dates.items():
    for day_name, test_start in days.items():

        print("=" * 80)
        print(f"Running Moirai2 SB - {season} - {day_name}: {test_start}")

        res = run_one_forecast(season, day_name, test_start)

        if res is not None:
            all_results.append(res)

            day_path = OUTPUT_DIR / f"moirai2_5min_SB_{season}_{day_name}.csv"
            res.to_csv(day_path, index=False)

            print(f"Saved: {day_path}")

if len(all_results) == 0:
    raise RuntimeError("No forecasts were produced. Check forecast_dates and data coverage.")

results_df = pd.concat(all_results, ignore_index=True)

all_path = OUTPUT_DIR / "moirai2_5min_SB_ALL_predictions.csv"
results_df.to_csv(all_path, index=False)

print("Done.")
print(f"Saved combined predictions: {all_path}")
print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")

# All features

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time

from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module

# ============================================================
# PATHS
# ============================================================
LOAD_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv"
WEATHER_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\meteo_imputed.csv"
DATES_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json"

OUTPUT_DIR = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\moirai2_5min_engineered"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
TARGET_COL = "avg_power"

BASE_WEATHER_COLS = [
    "Temp__C",
    "Hum__",
    "AvgWindSpeed_M_s",
    "Rain_Mm",
    "SolarRad_W_m_2",
]

PRED_LEN = 288
CONTEXT_LEN = 288 * 7
FREQ = "5min"

MODEL_NAME = "Salesforce/moirai-2.0-R-small"
BATCH_SIZE = 1

# ============================================================
# LOAD DATA
# ============================================================
load_df = pd.read_csv(LOAD_PATH)
weather_df = pd.read_csv(WEATHER_PATH)

load_df.columns = load_df.columns.str.strip()
weather_df.columns = weather_df.columns.str.strip()

weather_df = weather_df.rename(columns={"Date_Time": "timestamp"})

load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])

load_df = load_df.set_index("timestamp")
weather_df = weather_df.set_index("timestamp")

load_5min = load_df.resample(FREQ).mean()
weather_5min = weather_df.resample(FREQ).mean()

df = load_5min.join(weather_5min, how="inner").reset_index()
df = df.sort_values("timestamp").reset_index(drop=True)

# ============================================================
# ENGINEERED FEATURES
# ============================================================
def add_engineered_features(data, target_col):
    data = data.copy()
    ts = data["timestamp"]

    data["hour"] = ts.dt.hour
    data["minute"] = ts.dt.minute
    data["dayofweek"] = ts.dt.dayofweek
    data["dayofyear"] = ts.dt.dayofyear
    data["month"] = ts.dt.month
    data["is_weekend"] = (data["dayofweek"] >= 5).astype(int)

    steps_per_day = 288
    steps_per_week = 288 * 7

    step_of_day = data["hour"] * 12 + data["minute"] // 5
    step_of_week = data["dayofweek"] * steps_per_day + step_of_day

    for k in [1, 2, 3]:
        data[f"sin_day_{k}"] = np.sin(2 * np.pi * k * step_of_day / steps_per_day)
        data[f"cos_day_{k}"] = np.cos(2 * np.pi * k * step_of_day / steps_per_day)

    for k in [1, 2]:
        data[f"sin_week_{k}"] = np.sin(2 * np.pi * k * step_of_week / steps_per_week)
        data[f"cos_week_{k}"] = np.cos(2 * np.pi * k * step_of_week / steps_per_week)

    lag_steps = [
        1,
        2,
        3,
        6,
        12,
        24,
        288,
        288 * 2,
        288 * 7,
    ]

    for lag in lag_steps:
        data[f"{target_col}_lag_{lag}"] = data[target_col].shift(lag)

    rolling_windows = [
        3,
        12,
        288,
        288 * 7,
    ]

    shifted = data[target_col].shift(1)

    for window in rolling_windows:
        data[f"{target_col}_roll_mean_{window}"] = shifted.rolling(window).mean()
        data[f"{target_col}_roll_std_{window}"] = shifted.rolling(window).std()

    data["temp_sq"] = data["Temp__C"] ** 2
    data["heating_degree_18"] = np.maximum(18 - data["Temp__C"], 0)
    data["cooling_degree_22"] = np.maximum(data["Temp__C"] - 22, 0)
    data["is_daylight"] = (data["SolarRad_W_m_2"] > 10).astype(int)

    return data


df = add_engineered_features(df, TARGET_COL)

# ============================================================
# FEATURE LIST
# ============================================================
ENGINEERED_COLS = [
    "hour",
    "minute",
    "dayofweek",
    "dayofyear",
    "month",
    "is_weekend",

    "sin_day_1", "cos_day_1",
    "sin_day_2", "cos_day_2",
    "sin_day_3", "cos_day_3",

    "sin_week_1", "cos_week_1",
    "sin_week_2", "cos_week_2",

    f"{TARGET_COL}_lag_1",
    f"{TARGET_COL}_lag_2",
    f"{TARGET_COL}_lag_3",
    f"{TARGET_COL}_lag_6",
    f"{TARGET_COL}_lag_12",
    f"{TARGET_COL}_lag_24",
#    f"{TARGET_COL}_lag_288",
#    f"{TARGET_COL}_lag_576",
#    f"{TARGET_COL}_lag_2016",

    f"{TARGET_COL}_roll_mean_3",
    f"{TARGET_COL}_roll_std_3",
    f"{TARGET_COL}_roll_mean_12",
    f"{TARGET_COL}_roll_std_12",
    f"{TARGET_COL}_roll_mean_288",
#    f"{TARGET_COL}_roll_std_288",
#    f"{TARGET_COL}_roll_mean_2016",
#    f"{TARGET_COL}_roll_std_2016",

    "temp_sq",
    "heating_degree_18",
    "cooling_degree_22",
    "is_daylight",
]

FEATURE_COLS = BASE_WEATHER_COLS + ENGINEERED_COLS

df = df.dropna(subset=[TARGET_COL] + FEATURE_COLS).reset_index(drop=True)

print("Final feature columns:")
print(FEATURE_COLS)
print("Number of features:", len(FEATURE_COLS))
print("Data range:", df["timestamp"].min(), "to", df["timestamp"].max())

# ============================================================
# FORECAST DATES
# ============================================================
with open(DATES_PATH, "r") as f:
    forecast_dates = json.load(f)

# ============================================================
# LOAD MOIRAI2
# ============================================================
model = Moirai2Forecast(
    module=Moirai2Module.from_pretrained(MODEL_NAME),
    prediction_length=PRED_LEN,
    context_length=CONTEXT_LEN,
    target_dim=1,
    feat_dynamic_real_dim=0,
    past_feat_dynamic_real_dim=len(FEATURE_COLS),
)

predictor = model.create_predictor(batch_size=BATCH_SIZE)

# ============================================================
# HELPERS
# ============================================================
def extract_forecast_values(forecast):
    if hasattr(forecast, "quantile"):
        try:
            return np.asarray(forecast.quantile(0.5))[:PRED_LEN]
        except Exception:
            pass

        try:
            return np.asarray(forecast.quantile("0.5"))[:PRED_LEN]
        except Exception:
            pass

    if hasattr(forecast, "median"):
        return np.asarray(forecast.median)[:PRED_LEN]

    if hasattr(forecast, "mean"):
        return np.asarray(forecast.mean)[:PRED_LEN]

    if hasattr(forecast, "samples"):
        return np.median(np.asarray(forecast.samples), axis=0)[:PRED_LEN]

    raise ValueError(f"Cannot extract forecast values from object: {type(forecast)}")


def make_moirai_dataset(history_df):
    target = history_df[TARGET_COL].astype(np.float32).values

    past_features = history_df[FEATURE_COLS].astype(np.float32).values.T
    # shape: [num_features, time]

    entry = {
        "start": history_df["timestamp"].iloc[0],
        "target": target,
        "past_feat_dynamic_real": past_features,
    }

    return ListDataset([entry], freq=FREQ)

# ============================================================
# FORECAST FUNCTION
# ============================================================
def run_one_forecast(season, day_name, test_start):

    test_start = pd.Timestamp(test_start)
    test_end = test_start + pd.Timedelta(days=1)

    history_df = df[df["timestamp"] < test_start].tail(CONTEXT_LEN).copy()

    future_df = df[
        (df["timestamp"] >= test_start)
        & (df["timestamp"] < test_end)
    ].copy()

    if len(history_df) == 0:
        print(f"Skipping {season}-{day_name}: no history before {test_start}")
        return None

    if len(future_df) != PRED_LEN:
        print(f"Skipping {season}-{day_name}: expected {PRED_LEN}, got {len(future_df)}")
        return None

    dataset = make_moirai_dataset(history_df)

    forecasts = list(predictor.predict(dataset))

    if len(forecasts) != 1:
        raise RuntimeError(f"Expected 1 forecast, got {len(forecasts)}")

    pred_values = extract_forecast_values(forecasts[0])

    out = future_df[["timestamp", TARGET_COL]].copy()
    out = out.rename(columns={TARGET_COL: "actual"})

    out["prediction"] = pred_values
    out["season"] = season
    out["day"] = day_name

    return out

# ============================================================
# RUN ALL DAYS
# ============================================================
all_results = []

start_time = time.time()

for season, days in forecast_dates.items():
    for day_name, test_start in days.items():

        print("=" * 80)
        print(f"Running Moirai2 AF - {season} - {day_name}: {test_start}")

        res = run_one_forecast(season, day_name, test_start)

        if res is not None:
            all_results.append(res)

            day_path = OUTPUT_DIR / f"moirai2_5min_AF_{season}_{day_name}.csv"
            res.to_csv(day_path, index=False)

            print(f"Saved: {day_path}")

if len(all_results) == 0:
    raise RuntimeError("No forecasts were produced. Check forecast_dates and data coverage.")

results_df = pd.concat(all_results, ignore_index=True)

all_path = OUTPUT_DIR / "moirai2_5min_AF_predictions.csv"
results_df.to_csv(all_path, index=False)

print(f"Saved combined predictions: {all_path}")
print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")

# RFFS

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time

from sklearn.ensemble import RandomForestRegressor
from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module

# ============================================================
# PATHS
# ============================================================
LOAD_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv"
WEATHER_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\meteo_imputed.csv"
DATES_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json"

OUTPUT_DIR = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\moirai2_5min_engineered_RF95"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
TARGET_COL = "avg_power"

BASE_WEATHER_COLS = [
    "Temp__C",
    "Hum__",
    "AvgWindSpeed_M_s",
    "Rain_Mm",
    "SolarRad_W_m_2",
]

PRED_LEN = 288
CONTEXT_LEN = 288 * 7
FREQ = "5min"

RF_IMPORTANCE_COVERAGE = 0.95

MODEL_NAME = "Salesforce/moirai-2.0-R-small"
BATCH_SIZE = 1

# ============================================================
# LOAD DATA
# ============================================================
load_df = pd.read_csv(LOAD_PATH)
weather_df = pd.read_csv(WEATHER_PATH)

load_df.columns = load_df.columns.str.strip()
weather_df.columns = weather_df.columns.str.strip()

weather_df = weather_df.rename(columns={"Date_Time": "timestamp"})

load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])

load_df = load_df.set_index("timestamp")
weather_df = weather_df.set_index("timestamp")

load_5min = load_df.resample(FREQ).mean()
weather_5min = weather_df.resample(FREQ).mean()

df = load_5min.join(weather_5min, how="inner").reset_index()
df = df.sort_values("timestamp").reset_index(drop=True)

# ============================================================
# ENGINEERED FEATURES
# ============================================================
def add_engineered_features(data, target_col):
    data = data.copy()
    ts = data["timestamp"]

    data["hour"] = ts.dt.hour
    data["minute"] = ts.dt.minute
    data["dayofweek"] = ts.dt.dayofweek
    data["dayofyear"] = ts.dt.dayofyear
    data["month"] = ts.dt.month
    data["is_weekend"] = (data["dayofweek"] >= 5).astype(int)

    steps_per_day = 288
    steps_per_week = 288 * 7

    step_of_day = data["hour"] * 12 + data["minute"] // 5
    step_of_week = data["dayofweek"] * steps_per_day + step_of_day

    for k in [1, 2, 3]:
        data[f"sin_day_{k}"] = np.sin(2 * np.pi * k * step_of_day / steps_per_day)
        data[f"cos_day_{k}"] = np.cos(2 * np.pi * k * step_of_day / steps_per_day)

    for k in [1, 2]:
        data[f"sin_week_{k}"] = np.sin(2 * np.pi * k * step_of_week / steps_per_week)
        data[f"cos_week_{k}"] = np.cos(2 * np.pi * k * step_of_week / steps_per_week)

    lag_steps = [
        1, 2, 3, 6, 12, 24,
        288,
        288 * 2,
        288 * 7,
    ]

    for lag in lag_steps:
        data[f"{target_col}_lag_{lag}"] = data[target_col].shift(lag)

    rolling_windows = [3, 12, 288, 288 * 7]
    shifted = data[target_col].shift(1)

    for window in rolling_windows:
        data[f"{target_col}_roll_mean_{window}"] = shifted.rolling(window).mean()
        data[f"{target_col}_roll_std_{window}"] = shifted.rolling(window).std()

    data["temp_sq"] = data["Temp__C"] ** 2
    data["heating_degree_18"] = np.maximum(18 - data["Temp__C"], 0)
    data["cooling_degree_22"] = np.maximum(data["Temp__C"] - 22, 0)
    data["is_daylight"] = (data["SolarRad_W_m_2"] > 10).astype(int)

    return data


df = add_engineered_features(df, TARGET_COL)

ENGINEERED_COLS = [
    "hour", "minute", "dayofweek", "dayofyear", "month", "is_weekend",

    "sin_day_1", "cos_day_1",
    "sin_day_2", "cos_day_2",
    "sin_day_3", "cos_day_3",

    "sin_week_1", "cos_week_1",
    "sin_week_2", "cos_week_2",

    f"{TARGET_COL}_lag_1",
    f"{TARGET_COL}_lag_2",
    f"{TARGET_COL}_lag_3",
    f"{TARGET_COL}_lag_6",
    f"{TARGET_COL}_lag_12",
    f"{TARGET_COL}_lag_24",
    f"{TARGET_COL}_lag_288",
    f"{TARGET_COL}_lag_576",
    f"{TARGET_COL}_lag_2016",

    f"{TARGET_COL}_roll_mean_3",
    f"{TARGET_COL}_roll_std_3",
    f"{TARGET_COL}_roll_mean_12",
    f"{TARGET_COL}_roll_std_12",
    f"{TARGET_COL}_roll_mean_288",
    f"{TARGET_COL}_roll_std_288",
    f"{TARGET_COL}_roll_mean_2016",
    f"{TARGET_COL}_roll_std_2016",

    "temp_sq",
    "heating_degree_18",
    "cooling_degree_22",
    "is_daylight",
]

ALL_FEATURE_COLS = BASE_WEATHER_COLS + ENGINEERED_COLS

df = df.dropna(subset=[TARGET_COL] + ALL_FEATURE_COLS).reset_index(drop=True)

print("Total candidate features:", len(ALL_FEATURE_COLS))
print("Data range:", df["timestamp"].min(), "to", df["timestamp"].max())

# ============================================================
# FORECAST DATES
# ============================================================
with open(DATES_PATH, "r") as f:
    forecast_dates = json.load(f)

# ============================================================
# RANDOM FOREST 95% IMPORTANCE SELECTION
# ============================================================
def select_rf_95_features(train_df, feature_cols, target_col, season, day_name):
    X = train_df[feature_cols].copy()
    y = train_df[target_col].copy()

    rf = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )

    rf.fit(X, y)

    importance_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": rf.feature_importances_,
    })

    importance_df = importance_df.sort_values(
        "importance", ascending=False
    ).reset_index(drop=True)

    total_importance = importance_df["importance"].sum()

    if total_importance == 0:
        selected_features = feature_cols
        importance_df["importance_share"] = 0.0
        importance_df["cumulative_importance"] = 0.0
    else:
        importance_df["importance_share"] = (
            importance_df["importance"] / total_importance
        )

        importance_df["cumulative_importance"] = (
            importance_df["importance_share"].cumsum()
        )

        selected_features = importance_df.loc[
            importance_df["cumulative_importance"] <= RF_IMPORTANCE_COVERAGE,
            "feature"
        ].tolist()

        if len(selected_features) < len(importance_df):
            crossing_feature = importance_df.iloc[len(selected_features)]["feature"]
            selected_features.append(crossing_feature)

    if len(selected_features) == 0:
        selected_features = importance_df.head(1)["feature"].tolist()

    importance_df["selected"] = importance_df["feature"].isin(selected_features)
    importance_df["season"] = season
    importance_df["day"] = day_name

    return selected_features, importance_df

# ============================================================
# HELPERS
# ============================================================
def extract_forecast_values(forecast):
    if hasattr(forecast, "quantile"):
        try:
            return np.asarray(forecast.quantile(0.5))[:PRED_LEN]
        except Exception:
            pass

        try:
            return np.asarray(forecast.quantile("0.5"))[:PRED_LEN]
        except Exception:
            pass

    if hasattr(forecast, "median"):
        return np.asarray(forecast.median)[:PRED_LEN]

    if hasattr(forecast, "mean"):
        return np.asarray(forecast.mean)[:PRED_LEN]

    if hasattr(forecast, "samples"):
        return np.median(np.asarray(forecast.samples), axis=0)[:PRED_LEN]

    raise ValueError(f"Cannot extract forecast values from object: {type(forecast)}")


def make_moirai_dataset(history_df, selected_features):
    target = history_df[TARGET_COL].astype(np.float32).values

    past_features = history_df[selected_features].astype(np.float32).values.T

    entry = {
        "start": history_df["timestamp"].iloc[0],
        "target": target,
        "past_feat_dynamic_real": past_features,
    }

    return ListDataset([entry], freq=FREQ)


def make_predictor(n_selected_features):
    model = Moirai2Forecast(
        module=Moirai2Module.from_pretrained(MODEL_NAME),
        prediction_length=PRED_LEN,
        context_length=CONTEXT_LEN,
        target_dim=1,
        feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=n_selected_features,
    )

    return model.create_predictor(batch_size=BATCH_SIZE)

# ============================================================
# FORECAST FUNCTION
# ============================================================
def run_one_forecast(season, day_name, test_start):

    test_start = pd.Timestamp(test_start)
    test_end = test_start + pd.Timedelta(days=1)

    train_for_selection_df = df[df["timestamp"] < test_start].copy()
    history_df = df[df["timestamp"] < test_start].tail(CONTEXT_LEN).copy()

    future_df = df[
        (df["timestamp"] >= test_start)
        & (df["timestamp"] < test_end)
    ].copy()

    if len(train_for_selection_df) == 0:
        print(f"Skipping {season}-{day_name}: no training data before {test_start}")
        return None, None, None

    if len(history_df) == 0:
        print(f"Skipping {season}-{day_name}: no history before {test_start}")
        return None, None, None

    if len(future_df) != PRED_LEN:
        print(f"Skipping {season}-{day_name}: expected {PRED_LEN}, got {len(future_df)}")
        return None, None, None

    selected_features, importance_df = select_rf_95_features(
        train_df=train_for_selection_df,
        feature_cols=ALL_FEATURE_COLS,
        target_col=TARGET_COL,
        season=season,
        day_name=day_name,
    )

    print(f"Selected {len(selected_features)} / {len(ALL_FEATURE_COLS)} features")
    print(selected_features)

    predictor = make_predictor(len(selected_features))
    dataset = make_moirai_dataset(history_df, selected_features)

    forecasts = list(predictor.predict(dataset))

    if len(forecasts) != 1:
        raise RuntimeError(f"Expected 1 forecast, got {len(forecasts)}")

    pred_values = extract_forecast_values(forecasts[0])

    out = future_df[["timestamp", TARGET_COL]].copy()
    out = out.rename(columns={TARGET_COL: "actual"})

    out["prediction"] = pred_values
    out["season"] = season
    out["day"] = day_name
    out["n_selected_features"] = len(selected_features)

    selected_features_df = pd.DataFrame({
        "season": season,
        "day": day_name,
        "selected_feature": selected_features,
    })

    return out, importance_df, selected_features_df

# ============================================================
# RUN ALL DAYS
# ============================================================
all_results = []
all_importances = []
all_selected_features = []

start_time = time.time()

for season, days in forecast_dates.items():
    for day_name, test_start in days.items():

        print("=" * 80)
        print(f"Running Moirai2 RF95 - {season} - {day_name}: {test_start}")

        res, importance_df, selected_features_df = run_one_forecast(
            season, day_name, test_start
        )

        if res is not None:
            all_results.append(res)
            all_importances.append(importance_df)
            all_selected_features.append(selected_features_df)

            day_path = OUTPUT_DIR / f"moirai2_5min_RFFS_{season}_{day_name}.csv"
            res.to_csv(day_path, index=False)

            imp_path = OUTPUT_DIR / f"RF95_importance_{season}_{day_name}.csv"
            importance_df.to_csv(imp_path, index=False)

            selected_path = OUTPUT_DIR / f"RF95_selected_features_{season}_{day_name}.csv"
            selected_features_df.to_csv(selected_path, index=False)

            print(f"Saved prediction: {day_path}")
            print(f"Saved importance: {imp_path}")
            print(f"Saved selected features: {selected_path}")

if len(all_results) == 0:
    raise RuntimeError("No forecasts were produced. Check forecast_dates and data coverage.")

# ============================================================
# SAVE COMBINED FILES
# ============================================================
results_df = pd.concat(all_results, ignore_index=True)
importance_all_df = pd.concat(all_importances, ignore_index=True)
selected_all_df = pd.concat(all_selected_features, ignore_index=True)

results_df.to_csv(
    OUTPUT_DIR / "moirai2_5min_RF95_ALL_predictions.csv",
    index=False
)

importance_all_df.to_csv(
    OUTPUT_DIR / "RF95_ALL_importances.csv",
    index=False
)

selected_all_df.to_csv(
    OUTPUT_DIR / "RF95_ALL_selected_features.csv",
    index=False
)

print("Done.")
print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")
print(f"Output folder: {OUTPUT_DIR}")

c:\Users\Dimitrios\anaconda3\envs\moirai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total candidate features: 42
Data range: 2025-01-08 00:00:00 to 2025-12-31 23:55:00
Running Moirai2 RF95 - winter - day1: 2025-01-31 00:00:00
Selected 14 / 42 features
['avg_power_lag_1', 'avg_power_roll_mean_3', 'avg_power_lag_2', 'avg_power_roll_mean_12', 'avg_power_lag_2016', 'avg_power_roll_mean_288', 'avg_power_lag_3', 'avg_power_lag_6', 'sin_day_1', 'avg_power_lag_288', 'avg_power_lag_576', 'avg_power_roll_std_12', 'avg_power_roll_std_3', 'avg_power_roll_mean_2016']


Saved prediction: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\moirai2_5min_engineered_RF95\moirai2_5min_RFFS_winter_day1.csv
Saved importance: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\moirai2_5min_engineered_RF95\RF95_importance_winter_day1.csv
Saved selected features: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\moirai2_5min_engineered_RF95\RF95_selected_features_winter_day1.csv
Running Moirai2 RF95 - winter - day2: 2025-02-15 00:00:00
Selected 16 / 42 features
['avg_power_lag_1', 'avg_power_lag_2', 'avg_power_roll_mean_3', 'avg_power_lag_2016', 'avg_power_roll_mean_12', 'avg_power_lag_288', 'avg_power_roll_std_12', 'avg_power_lag_6', 'avg_power_lag_576', 'sin_day_1', 'avg_power_lag_3', 'avg_power_lag_12', 'avg_power_roll_mean_288', 'sin_day

# FSWL

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module

# ============================================================
# PATHS
# ============================================================
LOAD_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv"
WEATHER_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\meteo_imputed.csv"
DATES_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json"

OUTPUT_DIR = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\moirai2_5min_engineered_LASSO"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
TARGET_COL = "avg_power"

BASE_WEATHER_COLS = [
    "Temp__C",
    "Hum__",
    "AvgWindSpeed_M_s",
    "Rain_Mm",
    "SolarRad_W_m_2",
]

PRED_LEN = 288
CONTEXT_LEN = 288 * 7
FREQ = "5min"

LASSO_COEF_THRESHOLD = 1e-6


MODEL_NAME = "Salesforce/moirai-2.0-R-small"
BATCH_SIZE = 1

# ============================================================
# LOAD DATA
# ============================================================
load_df = pd.read_csv(LOAD_PATH)
weather_df = pd.read_csv(WEATHER_PATH)

load_df.columns = load_df.columns.str.strip()
weather_df.columns = weather_df.columns.str.strip()

weather_df = weather_df.rename(columns={"Date_Time": "timestamp"})

load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])

load_df = load_df.set_index("timestamp")
weather_df = weather_df.set_index("timestamp")

load_5min = load_df.resample(FREQ).mean()
weather_5min = weather_df.resample(FREQ).mean()

df = load_5min.join(weather_5min, how="inner").reset_index()
df = df.sort_values("timestamp").reset_index(drop=True)

# ============================================================
# ENGINEERED FEATURES
# ============================================================
def add_engineered_features(data, target_col):
    data = data.copy()
    ts = data["timestamp"]

    data["hour"] = ts.dt.hour
    data["minute"] = ts.dt.minute
    data["dayofweek"] = ts.dt.dayofweek
    data["dayofyear"] = ts.dt.dayofyear
    data["month"] = ts.dt.month
    data["is_weekend"] = (data["dayofweek"] >= 5).astype(int)

    steps_per_day = 288
    steps_per_week = 288 * 7

    step_of_day = data["hour"] * 12 + data["minute"] // 5
    step_of_week = data["dayofweek"] * steps_per_day + step_of_day

    for k in [1, 2, 3]:
        data[f"sin_day_{k}"] = np.sin(2 * np.pi * k * step_of_day / steps_per_day)
        data[f"cos_day_{k}"] = np.cos(2 * np.pi * k * step_of_day / steps_per_day)

    for k in [1, 2]:
        data[f"sin_week_{k}"] = np.sin(2 * np.pi * k * step_of_week / steps_per_week)
        data[f"cos_week_{k}"] = np.cos(2 * np.pi * k * step_of_week / steps_per_week)

    lag_steps = [1, 2, 3, 6, 12, 24, 288, 576, 2016]

    for lag in lag_steps:
        data[f"{target_col}_lag_{lag}"] = data[target_col].shift(lag)

    rolling_windows = [3, 12, 288, 2016]
    shifted = data[target_col].shift(1)

    for window in rolling_windows:
        data[f"{target_col}_roll_mean_{window}"] = shifted.rolling(window).mean()
        data[f"{target_col}_roll_std_{window}"] = shifted.rolling(window).std()

    data["temp_sq"] = data["Temp__C"] ** 2
    data["heating_degree_18"] = np.maximum(18 - data["Temp__C"], 0)
    data["cooling_degree_22"] = np.maximum(data["Temp__C"] - 22, 0)
    data["is_daylight"] = (data["SolarRad_W_m_2"] > 10).astype(int)

    return data


df = add_engineered_features(df, TARGET_COL)

ENGINEERED_COLS = [
    "hour", "minute", "dayofweek", "dayofyear", "month", "is_weekend",

    "sin_day_1", "cos_day_1",
    "sin_day_2", "cos_day_2",
    "sin_day_3", "cos_day_3",

    "sin_week_1", "cos_week_1",
    "sin_week_2", "cos_week_2",

    f"{TARGET_COL}_lag_1",
    f"{TARGET_COL}_lag_2",
    f"{TARGET_COL}_lag_3",
    f"{TARGET_COL}_lag_6",
    f"{TARGET_COL}_lag_12",
    f"{TARGET_COL}_lag_24",
    f"{TARGET_COL}_lag_288",
    f"{TARGET_COL}_lag_576",
    f"{TARGET_COL}_lag_2016",

    f"{TARGET_COL}_roll_mean_3",
    f"{TARGET_COL}_roll_std_3",
    f"{TARGET_COL}_roll_mean_12",
    f"{TARGET_COL}_roll_std_12",
    f"{TARGET_COL}_roll_mean_288",
    f"{TARGET_COL}_roll_std_288",
    f"{TARGET_COL}_roll_mean_2016",
    f"{TARGET_COL}_roll_std_2016",

    "temp_sq",
    "heating_degree_18",
    "cooling_degree_22",
    "is_daylight",
]

ALL_FEATURE_COLS = BASE_WEATHER_COLS + ENGINEERED_COLS

df = df.dropna(subset=[TARGET_COL] + ALL_FEATURE_COLS).reset_index(drop=True)

print("Total candidate features:", len(ALL_FEATURE_COLS))
print("Data range:", df["timestamp"].min(), "to", df["timestamp"].max())

# ============================================================
# FORECAST DATES
# ============================================================
with open(DATES_PATH, "r") as f:
    forecast_dates = json.load(f)

# ============================================================
# LASSO FEATURE SELECTION
# ============================================================
def select_lasso_features(train_df, feature_cols, target_col, season, day_name):
    X = train_df[feature_cols].copy()
    y = train_df[target_col].copy()

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", LassoCV(
            alphas=np.logspace(-4, 2, 40),
            cv=5,
            max_iter=20000,
            random_state=42,
            n_jobs=-1,
        ))
    ])

    model.fit(X, y)

    lasso = model.named_steps["lasso"]
    coefs = lasso.coef_

    coef_df = pd.DataFrame({
        "feature": feature_cols,
        "coefficient": coefs,
        "abs_coefficient": np.abs(coefs),
    })

    coef_df = coef_df.sort_values(
        "abs_coefficient", ascending=False
    ).reset_index(drop=True)

    selected_features = coef_df.loc[
        coef_df["abs_coefficient"] > LASSO_COEF_THRESHOLD,
        "feature"
    ].tolist()

    if len(selected_features) == 0:
        selected_features = coef_df.head(1)["feature"].tolist()

    coef_df["selected"] = coef_df["feature"].isin(selected_features)
    coef_df["season"] = season
    coef_df["day"] = day_name
    coef_df["lasso_alpha"] = lasso.alpha_

    return selected_features, coef_df

# ============================================================
# HELPERS
# ============================================================
def extract_forecast_values(forecast):
    if hasattr(forecast, "quantile"):
        try:
            return np.asarray(forecast.quantile(0.5))[:PRED_LEN]
        except Exception:
            pass

        try:
            return np.asarray(forecast.quantile("0.5"))[:PRED_LEN]
        except Exception:
            pass

    if hasattr(forecast, "median"):
        return np.asarray(forecast.median)[:PRED_LEN]

    if hasattr(forecast, "mean"):
        return np.asarray(forecast.mean)[:PRED_LEN]

    if hasattr(forecast, "samples"):
        return np.median(np.asarray(forecast.samples), axis=0)[:PRED_LEN]

    raise ValueError(f"Cannot extract forecast values from object: {type(forecast)}")


def make_moirai_dataset(history_df, selected_features):
    target = history_df[TARGET_COL].astype(np.float32).values

    past_features = history_df[selected_features].astype(np.float32).values.T

    entry = {
        "start": history_df["timestamp"].iloc[0],
        "target": target,
        "past_feat_dynamic_real": past_features,
    }

    return ListDataset([entry], freq=FREQ)


def make_predictor(n_selected_features):
    model = Moirai2Forecast(
        module=Moirai2Module.from_pretrained(MODEL_NAME),
        prediction_length=PRED_LEN,
        context_length=CONTEXT_LEN,
        target_dim=1,
        feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=n_selected_features,
    )

    return model.create_predictor(batch_size=BATCH_SIZE)

# ============================================================
# FORECAST FUNCTION
# ============================================================
def run_one_forecast(season, day_name, test_start):

    test_start = pd.Timestamp(test_start)
    test_end = test_start + pd.Timedelta(days=1)

    train_for_selection_df = df[df["timestamp"] < test_start].copy()
    history_df = df[df["timestamp"] < test_start].tail(CONTEXT_LEN).copy()

    future_df = df[
        (df["timestamp"] >= test_start)
        & (df["timestamp"] < test_end)
    ].copy()

    if len(train_for_selection_df) == 0:
        print(f"Skipping {season}-{day_name}: no training data before {test_start}")
        return None, None, None

    if len(history_df) == 0:
        print(f"Skipping {season}-{day_name}: no history before {test_start}")
        return None, None, None

    if len(future_df) != PRED_LEN:
        print(f"Skipping {season}-{day_name}: expected {PRED_LEN}, got {len(future_df)}")
        return None, None, None

    selected_features, coef_df = select_lasso_features(
        train_df=train_for_selection_df,
        feature_cols=ALL_FEATURE_COLS,
        target_col=TARGET_COL,
        season=season,
        day_name=day_name,
    )

    print(f"Selected {len(selected_features)} / {len(ALL_FEATURE_COLS)} features")
    print(selected_features)

    predictor = make_predictor(len(selected_features))
    dataset = make_moirai_dataset(history_df, selected_features)

    forecasts = list(predictor.predict(dataset))

    if len(forecasts) != 1:
        raise RuntimeError(f"Expected 1 forecast, got {len(forecasts)}")

    pred_values = extract_forecast_values(forecasts[0])

    out = future_df[["timestamp", TARGET_COL]].copy()
    out = out.rename(columns={TARGET_COL: "actual"})

    out["prediction"] = pred_values
    out["season"] = season
    out["day"] = day_name
    out["n_selected_features"] = len(selected_features)

    selected_features_df = pd.DataFrame({
        "season": season,
        "day": day_name,
        "selected_feature": selected_features,
    })

    return out, coef_df, selected_features_df

# ============================================================
# RUN ALL DAYS
# ============================================================
all_results = []
all_lasso_coefs = []
all_selected_features = []

start_time = time.time()

for season, days in forecast_dates.items():
    for day_name, test_start in days.items():

        print("=" * 80)
        print(f"Running Moirai2 LASSO - {season} - {day_name}: {test_start}")

        res, coef_df, selected_features_df = run_one_forecast(
            season, day_name, test_start
        )

        if res is not None:
            all_results.append(res)
            all_lasso_coefs.append(coef_df)
            all_selected_features.append(selected_features_df)

            day_path = OUTPUT_DIR / f"moirai2_5min_FSWL_{season}_{day_name}.csv"
            res.to_csv(day_path, index=False)

            coef_path = OUTPUT_DIR / f"LASSO_coefficients_{season}_{day_name}.csv"
            coef_df.to_csv(coef_path, index=False)

            selected_path = OUTPUT_DIR / f"LASSO_selected_features_{season}_{day_name}.csv"
            selected_features_df.to_csv(selected_path, index=False)

            print(f"Saved prediction: {day_path}")
            print(f"Saved coefficients: {coef_path}")
            print(f"Saved selected features: {selected_path}")

if len(all_results) == 0:
    raise RuntimeError("No forecasts were produced. Check forecast_dates and data coverage.")

# ============================================================
# SAVE COMBINED FILES
# ============================================================
results_df = pd.concat(all_results, ignore_index=True)
lasso_coefs_all_df = pd.concat(all_lasso_coefs, ignore_index=True)
selected_all_df = pd.concat(all_selected_features, ignore_index=True)

results_df.to_csv(
    OUTPUT_DIR / "moirai2_5min_LASSO_ALL_predictions.csv",
    index=False
)

lasso_coefs_all_df.to_csv(
    OUTPUT_DIR / "LASSO_ALL_coefficients.csv",
    index=False
)

selected_all_df.to_csv(
    OUTPUT_DIR / "LASSO_ALL_selected_features.csv",
    index=False
)

print("Done.")
print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")
print(f"Output folder: {OUTPUT_DIR}")

Total candidate features: 42
Data range: 2025-01-08 00:00:00 to 2025-12-31 23:55:00
Running Moirai2 LASSO - winter - day1: 2025-01-31 00:00:00
Selected 30 / 42 features
['avg_power_lag_1', 'avg_power_lag_2', 'avg_power_lag_3', 'avg_power_roll_mean_12', 'cos_day_3', 'avg_power_roll_std_12', 'avg_power_roll_std_3', 'avg_power_lag_24', 'avg_power_lag_576', 'cos_day_1', 'avg_power_lag_12', 'avg_power_lag_288', 'avg_power_lag_2016', 'cos_day_2', 'sin_week_2', 'avg_power_roll_std_2016', 'sin_day_2', 'cos_week_1', 'sin_day_3', 'avg_power_roll_std_288', 'cos_week_2', 'avg_power_roll_mean_2016', 'Hum__', 'is_daylight', 'sin_day_1', 'hour', 'Rain_Mm', 'avg_power_roll_mean_288', 'sin_week_1', 'dayofyear']
Saved prediction: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\moirai2_5min_engineered_LASSO\moirai2_5min_FSWL_winter_day1.csv
Saved coefficients: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-T

# Univariate

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time

from gluonts.dataset.common import ListDataset
from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module

# ============================================================
# PATHS
# ============================================================
LOAD_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv"
DATES_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json"

OUTPUT_DIR = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\moirai2_5min_univariate"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
TARGET_COL = "avg_power"

PRED_LEN = 288
CONTEXT_LEN = 288 * 7   # use 7 days to reduce RAM
FREQ = "5min"

MODEL_NAME = "Salesforce/moirai-2.0-R-small"
BATCH_SIZE = 1

# ============================================================
# LOAD LOAD DATA ONLY
# ============================================================
load_df = pd.read_csv(LOAD_PATH)
load_df.columns = load_df.columns.str.strip()

load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
load_df = load_df.set_index("timestamp")

load_5min = load_df.resample(FREQ).mean()

df = load_5min.reset_index()
df = df.sort_values("timestamp").reset_index(drop=True)
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

df[TARGET_COL] = df[TARGET_COL].astype("float32")

print("Data range:", df["timestamp"].min(), "to", df["timestamp"].max())
print(df.head())

# ============================================================
# FORECAST DATES
# ============================================================
with open(DATES_PATH, "r") as f:
    forecast_dates = json.load(f)

# ============================================================
# LOAD MOIRAI2
# ============================================================
model = Moirai2Forecast(
    module=Moirai2Module.from_pretrained(MODEL_NAME),
    prediction_length=PRED_LEN,
    context_length=CONTEXT_LEN,
    target_dim=1,
    feat_dynamic_real_dim=0,
    past_feat_dynamic_real_dim=0,
)

predictor = model.create_predictor(batch_size=BATCH_SIZE)

# ============================================================
# HELPERS
# ============================================================
def extract_forecast_values(forecast):
    if hasattr(forecast, "quantile"):
        try:
            return np.asarray(forecast.quantile(0.5))[:PRED_LEN]
        except Exception:
            pass

        try:
            return np.asarray(forecast.quantile("0.5"))[:PRED_LEN]
        except Exception:
            pass

    if hasattr(forecast, "median"):
        return np.asarray(forecast.median)[:PRED_LEN]

    if hasattr(forecast, "mean"):
        return np.asarray(forecast.mean)[:PRED_LEN]

    if hasattr(forecast, "samples"):
        return np.median(np.asarray(forecast.samples), axis=0)[:PRED_LEN]

    raise ValueError(f"Cannot extract forecast values from object: {type(forecast)}")


def make_moirai_dataset(history_df):
    target = history_df[TARGET_COL].astype(np.float32).values

    entry = {
        "start": history_df["timestamp"].iloc[0],
        "target": target,
    }

    return ListDataset([entry], freq=FREQ)

# ============================================================
# FORECAST FUNCTION
# ============================================================
def run_one_forecast(season, day_name, test_start):

    test_start = pd.Timestamp(test_start)
    test_end = test_start + pd.Timedelta(days=1)

    history_df = df[df["timestamp"] < test_start].tail(CONTEXT_LEN).copy()

    future_df = df[
        (df["timestamp"] >= test_start)
        & (df["timestamp"] < test_end)
    ].copy()

    if len(history_df) == 0:
        print(f"Skipping {season}-{day_name}: no history before {test_start}")
        return None

    if len(future_df) != PRED_LEN:
        print(f"Skipping {season}-{day_name}: expected {PRED_LEN}, got {len(future_df)}")
        return None

    dataset = make_moirai_dataset(history_df)

    forecasts = list(predictor.predict(dataset))

    if len(forecasts) != 1:
        raise RuntimeError(f"Expected 1 forecast, got {len(forecasts)}")

    pred_values = extract_forecast_values(forecasts[0])

    out = future_df[["timestamp", TARGET_COL]].copy()
    out = out.rename(columns={TARGET_COL: "actual"})

    out["prediction"] = pred_values
    out["season"] = season
    out["day"] = day_name

    return out

# ============================================================
# RUN ALL DAYS
# ============================================================
all_results = []

start_time = time.time()

for season, days in forecast_dates.items():
    for day_name, test_start in days.items():

        print("=" * 80)
        print(f"Running Moirai2 univariate - {season} - {day_name}: {test_start}")

        res = run_one_forecast(season, day_name, test_start)

        if res is not None:
            all_results.append(res)

            day_path = OUTPUT_DIR / f"moirai2_5min_univariate_{season}_{day_name}.csv"
            res.to_csv(day_path, index=False)

            print(f"Saved: {day_path}")

if len(all_results) == 0:
    raise RuntimeError("No forecasts were produced. Check forecast_dates and data coverage.")

results_df = pd.concat(all_results, ignore_index=True)

all_path = OUTPUT_DIR / "moirai2_5min_univariate_ALL_predictions.csv"
results_df.to_csv(all_path, index=False)

print("Done.")
print(f"Saved combined predictions: {all_path}")
print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")

Data range: 2025-01-01 00:00:00 to 2025-12-31 23:55:00
            timestamp    avg_power
0 2025-01-01 00:00:00  2954.242920
1 2025-01-01 00:05:00  2642.173096
2 2025-01-01 00:10:00  2646.241211
3 2025-01-01 00:15:00  2685.558105
4 2025-01-01 00:20:00  2665.809570
Running Moirai2 univariate - winter - day1: 2025-01-31 00:00:00
Saved: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\moirai2_5min_univariate\moirai2_5min_univariate_winter_day1.csv
Running Moirai2 univariate - winter - day2: 2025-02-15 00:00:00
Saved: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\moirai2_5min_univariate\moirai2_5min_univariate_winter_day2.csv
Running Moirai2 univariate - winter - day3: 2025-12-02 00:00:00
Saved: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\moirai2_5min_uni

# end